In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import os
# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# =============================================================================
# 1. CARGA Y CONCATENACIÓN DE DATOS
# =============================================================================

def cargar_y_concatenar(PATH, files):
    """
    Carga y concatena todos los archivos xls
    """
    datos_concatenados = pd.DataFrame()

    for i, filename in enumerate(files, 1):
        print(f"📄 Procesando archivo {i}/{len(files)}: {filename}")

        try:
            # Leer el archivo
            lectura = pd.read_excel(os.path.join(PATH, filename), sheet_name="Sheet1", skiprows=0)

            # Extraer año y semestre del nombre
            # Ajusta estas posiciones según tu nomenclatura
            try:
                lectura['Anho'] = int(filename[16:20])  # Ajusta según posición del año
                lectura['Semestre'] = int(filename[27])  # Ajusta según posición del semestre
            except:
                print(f"  ⚠️ No se pudo extraer año/semestre de {filename}")
                continue

            # Agregar nombre del archivo como referencia
            lectura['Archivo_Origen'] = filename

            datos_concatenados = pd.concat([datos_concatenados, lectura], axis=0, ignore_index=True)
            print(f"  ✅ Registros agregados: {len(lectura):,} | Total acumulado: {len(datos_concatenados):,}")

        except Exception as e:
            print(f"  ❌ Error al procesar {filename}: {e}")

    print(f"\n✅ Total de registros concatenados: {len(datos_concatenados):,}")
    return datos_concatenados

# =============================================================================
# 2. ANÁLISIS DESCRIPTIVO GENERAL
# =============================================================================

def analisis_descriptivo_general(df):
    """
    Realiza un análisis descriptivo completo de los datos
    """
    print("="*80)
    print("📊 ANÁLISIS DESCRIPTIVO GENERAL DE LOS DATOS")
    print("="*80)

    # 2.1. Información básica del dataset
    print("\n" + "="*80)
    print("1. INFORMACIÓN BÁSICA DEL DATASET")
    print("="*80)

    print(f"📌 Total de registros: {len(df):,}")
    print(f"📌 Total de columnas: {len(df.columns)}")
    print(f"📌 Total de estudiantes únicos: {df['ALUMNO_ID'].nunique():,}")
    print(f"📌 Total de asignaturas únicas: {df['Asignatura'].nunique():,}")
    print(f"📌 Rango de años: {df['Anho'].min()} - {df['Anho'].max()}")
    print(f"📌 Semestres disponibles: {sorted(df['Semestre'].unique())}")

    # 2.2. Estructura de los datos
    print("\n" + "="*80)
    print("2. ESTRUCTURA DE LOS DATOS")
    print("="*80)

    print("\n📋 Primeras 5 filas:")
    display(df.head())

    print("\n📋 Últimas 5 filas:")
    display(df.tail())

    print("\n📋 Tipos de datos por columna:")
    print(df.dtypes)

    # 2.3. Estadísticas descriptivas
    print("\n" + "="*80)
    print("3. ESTADÍSTICAS DESCRIPTIVAS")
    print("="*80)

    # Columnas numéricas
    columnas_numericas = df.select_dtypes(include=[np.number]).columns
    print("\n📊 Estadísticas de columnas numéricas:")
    display(df[columnas_numericas].describe())

    # Columnas categóricas
    columnas_categoricas = df.select_dtypes(include=['object']).columns
    print("\n📊 Columnas categóricas:")
    for col in columnas_categoricas[:10]:  # Mostrar primeras 10
        print(f"\n{col}:")
        print(f"  - Valores únicos: {df[col].nunique()}")
        print(f"  - Más frecuente: {df[col].mode().iloc[0] if not df[col].mode().empty else 'N/A'}")
        print(f"  - Frecuencia: {df[col].value_counts().iloc[0] if not df[col].empty else 0}")

    # 2.4. Análisis de datos faltantes
    print("\n" + "="*80)
    print("4. ANÁLISIS DE DATOS FALTANTES")
    print("="*80)

    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing_Count': missing_data,
        'Missing_Percent': missing_percent
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

    if len(missing_df) > 0:
        print(f"\n🔍 Columnas con datos faltantes ({len(missing_df)} columnas):")
        display(missing_df)
    else:
        print("✅ No hay datos faltantes en el dataset")

    # 2.5. Análisis de duplicados
    print("\n" + "="*80)
    print("5. ANÁLISIS DE REGISTROS DUPLICADOS")
    print("="*80)

    duplicados = df.duplicated().sum()
    print(f"📌 Registros duplicados: {duplicados:,} ({duplicados/len(df)*100:.2f}%) दिखनें")

    if duplicados > 0:
        print("\n🔍 Ejemplo de registros duplicados:")
        display(df[df.duplicated(keep=False)].head(5))

    # 2.6. Análisis por Año y Semestre
    print("\n" + "="*80)
    print("6. DISTRIBUCIÓN POR AÑO Y SEMESTRE")
    print("="*80)

    year_semester_counts = df.groupby(['Anho', 'Semestre']).size().reset_index(name='Registros')
    print("\n📊 Distribución de registros por año y semestre:")
    display(year_semester_counts)

    # Crear tabla pivote para mejor visualización
    pivot_counts = df.pivot_table(index='Anho', columns='Semestre', aggfunc='size', fill_value=0)
    print("\n📊 Tabla pivote de registros por año y semestre:")
    display(pivot_counts)

    # 2.7. Análisis de Asignaturas
    print("\n" + "="*80)
    print("7. ANÁLISIS DE ASIGNATURAS")
    print("="*80)

    # Top 10 asignaturas con más registros
    top_asignaturas = df['Asignatura'].value_counts().head(10)
    print("\n📚 Top 10 asignaturas con más registros:")
    display(top_asignaturas)

    # Asignaturas por nivel
    if 'Cod.Curso' in df.columns:
        asignaturas_por_nivel = df.groupby('Cod.Curso')['Asignatura'].nunique().sort_index()
        print("\n📚 Número de asignaturas por nivel:")
        print(asignaturas_por_nivel)

    # 2.8. Análisis de Carreras
    print("\n" + "="*80)
    print("8. ANÁLISIS DE CARRERAS")
    print("="*80)

    # Extraer carrera de Cod.Car.Sec
    if 'Cod.Car.Sec' in df.columns:
        df['Carrera'] = df['Cod.Car.Sec'].str.split('-').str[0]
        carreras_counts = df['Carrera'].value_counts()
        print("\n🎓 Distribución de estudiantes por carrera:")
        display(carreras_counts)

        # Verificar que son 7 carreras
        print(f"\n📌 Total de carreras únicas: {df['Carrera'].nunique()}")
        print(f"📌 Carreras encontradas:")
        for carrera in sorted(df['Carrera'].unique()):
            print(f"  - {carrera}")

    # 2.9. Análisis de Aprobación
    print("\n" + "="*80)
    print("9. ANÁLISIS DE APROBACIÓN")
    print("="*80)

    if 'Aprobado' in df.columns:
        aprobacion_counts = df['Aprobado'].value_counts()
        print("\n✅ Estado de aprobación:")
        display(aprobacion_counts)
        print(f"\n📊 Tasa de aprobación: {aprobacion_counts.get('S', 0)/len(df)*100:.2f}%")

        # Aprobación por año
        aprobacion_por_anho = df.groupby('Anho')['Aprobado'].value_counts(normalize=True).unstack()
        aprobacion_por_anho = aprobacion_por_anho * 100
        print("\n📈 Tasa de aprobación por año:")
        display(aprobacion_por_anho)

    # 2.10. Análisis de Notas
    print("\n" + "="*80)
    print("10. ANÁLISIS DE NOTAS")
    print("="*80)

    if 'Nota.Final' in df.columns:
        print(f"\n📊 Estadísticas de Nota Final:")
        print(df['Nota.Final'].describe())


    return {
        'df_info': df.info(),
        'missing_data': missing_df,
        'year_semester_counts': year_semester_counts,
        'carreras_counts': carreras_counts if 'Cod.Car.Sec' in df.columns else None,
        'aprobacion_counts': aprobacion_counts if 'Aprobado' in df.columns else None
    }

# =============================================================================
# 3. VISUALIZACIONES
# =============================================================================

def visualizar_datos(df):
    """
    Crea visualizaciones principales de los datos
    """
    print("\n" + "="*80)
    print("📈 VISUALIZACIONES DE DATOS")
    print("="*80)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Distribución por año
    if 'Anho' in df.columns:
        df['Anho'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='skyblue')
        axes[0,0].set_title('Distribución de Registros por Año')
        axes[0,0].set_xlabel('Año')
        axes[0,0].set_ylabel('Número de Registros')
        axes[0,0].tick_params(axis='x', rotation=45)

    # 2. Distribución por carrera
    if 'Carrera' in df.columns:
        df['Carrera'].value_counts().plot(kind='bar', ax=axes[0,1], color='lightgreen')
        axes[0,1].set_title('Distribución por Carrera')
        axes[0,1].set_xlabel('Carrera')
        axes[0,1].set_ylabel('Número de Registros')
        axes[0,1].tick_params(axis='x', rotation=45)

    # 3. Estado de aprobación
    if 'Aprobado' in df.columns:
        df['Aprobado'].value_counts().plot(kind='pie', ax=axes[0,2], autopct='%1.1f%%', colors=['lightcoral', 'lightblue'])
        axes[0,2].set_title('Estado de Aprobación')
        axes[0,2].set_ylabel('')

    # 4. Distribución de notas
    if 'Nota.Final' in df.columns:
        df['Nota.Final'].dropna().hist(bins=20, ax=axes[1,0], color='orange', edgecolor='black')
        axes[1,0].set_title('Distribución de Notas Finales')
        axes[1,0].set_xlabel('Nota Final')
        axes[1,0].set_ylabel('Frecuencia')

    # 5. Nota por año
    if 'Nota.Final' in df.columns and 'Anho' in df.columns:
        df.boxplot(column='Nota.Final', by='Anho', ax=axes[1,1])
        axes[1,1].set_title('Notas por Año')
        axes[1,1].set_xlabel('Año')
        axes[1,1].set_ylabel('Nota Final')

    # 6. Top 10 asignaturas
    df['Asignatura'].value_counts().head(10).plot(kind='barh', ax=axes[1,2], color='purple')
    axes[1,2].set_title('Top 10 Asignaturas con más Registros')
    axes[1,2].set_xlabel('Número de Registros')

    plt.tight_layout()
    plt.show()

    # Visualización adicional: Mapa de calor de datos faltantes
    print("\n📊 Mapa de calor de datos faltantes:")
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Mapa de Calor de Datos Faltantes')
    plt.show()

# =============================================================================
# 4. EJECUCIÓN PRINCIPAL
# =============================================================================

def main():
    """
    Función principal que ejecuta todo el análisis
    """
    # Configurar ruta
    # IMPORTANT: Change this PATH to where your files are located in Colab.
    # For example, if you upload a folder named 'data' to /content/ and 'reglamentonuevo' is inside it:
    PATH = "/content/data" # <--- ADJUST THIS LINE

    # Obtener archivos
    # Use os.path.join for cross-platform compatibility
    files = [f for f in os.listdir(os.path.join(PATH, "reglamentonuevo")) if f.endswith(('.xls', '.xlsx'))]

    if not files:
        print("❌ No se encontraron archivos XLS en la ruta especificada")
        return

    print(f"🔍 Encontrados {len(files)} archivos XLS para procesar")
    print(f"📂 Ruta: {PATH}")

    # 1. Cargar y concatenar datos
    print("\n" + "="*80)
    print("🔄 PROCESO DE CARGA Y CONCATENACIÓN")
    print("="*80)

    datos = cargar_y_concatenar(os.path.join(PATH, "reglamentonuevo"), files)

    if len(datos) == 0:
        print("❌ No se pudieron cargar datos")
        return

    # 2. Análisis descriptivo general
    print("\n" + "="*80)
    print("📊 ANÁLISIS DESCRIPTIVO GENERAL")
    print("="*80)

    resultados = analisis_descriptivo_general(datos)

    # 3. Visualizaciones
    visualizar_datos(datos)

    # 4. Guardar datos procesados
    datos.to_csv("datos_concatenados.csv", index=False)
    print("\n✅ Datos concatenados guardados en 'datos_concatenados.csv'")

    # 5. Resumen final
    print("\n" + "="*80)
    print("📋 RESUMEN FINAL")
    print("="*80)

    print(f"""
    ✅ PROCESO COMPLETADO EXITOSAMENTE

    📊 Estadísticas del dataset:
    • Registros totales: {len(datos):,}
    • Estudiantes únicos: {datos['ALUMNO_ID'].nunique():,}
    • Asignaturas únicas: {datos['Asignatura'].nunique():,}
    • Años académicos: {datos['Anho'].min()} - {datos['Anho'].max()}
    • Semestres: {', '.join(map(str, sorted(datos['Semestre'].unique())))}
    • Carreras: {datos['Carrera'].nunique() if 'Carrera' in datos.columns else 'N/A'}

    📁 Archivos generados:
    • datos_concatenados.csv
    """)

    return datos, resultados

# =============================================================================
# EJECUTAR EL ANÁLISIS
# =============================================================================

if __name__ == "__main__":
    datos_procesados, resultados_analisis = main()

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/reglamentonuevo'